# Skin Cancer Melanoma vs UV Index Analysis

This notebook explores the relationship between UV radiation exposure and skin cancer (melanoma) incidence across multiple countries with varying UV exposure levels.

## Overview
- **Countries Analyzed**: Australia (high UV), USA, Argentina, Uganda (equatorial), Sweden/Norway/Denmark (Nordic - low UV)
- **Focus**: Melanoma of skin incidence rates and their possible correlation with UV exposure

## Key Questions
1. Is there a correlation between average UV index and melanoma incidence?
2. How do melanoma rates differ between high-UV (Australia) and low-UV (Nordic) countries?
3. What temporal trends exist in melanoma cases over the years?

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import os
import sys

# Add parent directories to path for utils
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Libraries imported successfully!")
print(f"Project root: {project_root}")

## 1. Data Loading and Exploration

### 1.1 Define Data Paths

In [ ]:
# Define paths to data directories
cancer_data_path = project_root / "data" / "countries" / "cancer"
uv_data_path = project_root / "data" / "countries" / "uv"

# List all available cancer data files
cancer_files = list(cancer_data_path.glob("*.csv"))
print("Cancer Data Files Available:")
for f in cancer_files:
    print(f"   - {f.name}")

print(f"\nUV Data Files Available:")
uv_files = list(uv_data_path.glob("*.csv"))
for f in uv_files:
    print(f"   - {f.name}")

### 1.2 Load and Explore Cancer Data Structure

In [ ]:
# Load individual country cancer data files
country_cancer_files = ['Argentina.csv', 'Australia.csv', 'Sweden.csv', 'Uganda.csv', 'USA.csv']

cancer_data = {}
for file in country_cancer_files:
    file_path = cancer_data_path / file
    if file_path.exists():
        country_name = file.replace('.csv', '')
        cancer_data[country_name] = pd.read_csv(file_path)
        print(f"\n{'='*60}")
        print(f"{country_name} Cancer Data:")
        print(f"{'='*60}")
        print(f"Shape: {cancer_data[country_name].shape}")
        print(f"\nColumns: {list(cancer_data[country_name].columns)}")
        print(f"\nCancer types available:")
        print(cancer_data[country_name]['Cancer label'].unique())

In [ ]:
# Load Nordic countries melanoma data (separate format)
nordic_asr = pd.read_csv(cancer_data_path / 'norway_sweden_denmark_melanoma_of_skin_asr_world.csv')
nordic_numbers = pd.read_csv(cancer_data_path / 'norway_sweden_denmark_melanoma_of_skin_numbers.csv')

print("Nordic Countries Melanoma Data (ASR - Age Standardized Rate):")
print(nordic_asr.head())
print(f"\nShape: {nordic_asr.shape}")

print("\nNordic Countries Melanoma Data (Absolute Numbers):")
print(nordic_numbers.head())
print(f"\nShape: {nordic_numbers.shape}")

### 1.3 Extract Melanoma Data from All Countries

In [ ]:
# Extract melanoma data from each country
melanoma_data = {}

for country, df in cancer_data.items():
    # Filter for melanoma cases
    melanoma_df = df[df['Cancer label'] == 'Melanoma of skin'].copy()
    if not melanoma_df.empty:
        melanoma_data[country] = melanoma_df
        print(f"{country}: {len(melanoma_df)} years of melanoma data ({melanoma_df['Year'].min()}-{melanoma_df['Year'].max()})")
    else:
        print(f"Warning: {country}: No melanoma data found")

# Process Nordic data into same format
# Transform wide format to long format for Nordic countries
nordic_countries_asr = []
for country in ['Denmark', 'Norway', 'Sweden']:
    years = [col for col in nordic_asr.columns if col != 'Label']
    country_data = nordic_asr[nordic_asr['Label'] == country][years].T.reset_index()
    country_data.columns = ['Year', 'ASR (World)']
    country_data['Country label'] = country
    country_data['Year'] = pd.to_numeric(country_data['Year'], errors='coerce')
    country_data['ASR (World)'] = pd.to_numeric(country_data['ASR (World)'].replace('-', np.nan), errors='coerce')
    country_data = country_data.dropna(subset=['ASR (World)'])
    nordic_countries_asr.append(country_data)

nordic_asr_long = pd.concat(nordic_countries_asr, ignore_index=True)
print(f"Nordic countries: Melanoma ASR data from {nordic_asr_long['Year'].min():.0f}-{nordic_asr_long['Year'].max():.0f}")
print(nordic_asr_long.head(10))

## 2. UV Data Loading and Processing

### 2.1 Load UV Data and Calculate Yearly Averages

In [ ]:
# Country coordinates
COUNTRY_COORDINATES = {
    'Argentina': {'lat': -38.42, 'lon': -63.62, 'name': 'Argentina'},
    'Australia': {'lat': -25.27, 'lon': 133.78, 'name': 'Australia'},
    'USA': {'lat': 39.83, 'lon': -98.58, 'name': 'USA'},
    'Uganda': {'lat': 1.37, 'lon': 32.29, 'name': 'Uganda'},
    'Sweden': {'lat': 62.19, 'lon': 17.55, 'name': 'Sweden'},
    'Denmark': {'lat': 56.26, 'lon': 9.50, 'name': 'Denmark'},
    'Norway': {'lat': 60.47, 'lon': 8.47, 'name': 'Norway'}
}

# Load available UV data files
uv_data = {}
countries_with_uv = ['Argentina', 'Australia', 'USA', 'Uganda', 'Sweden']

for country in countries_with_uv:
    file_path = uv_data_path / f"{country}.csv"
    if file_path.exists():
        df = pd.read_csv(file_path)
        df['Date'] = pd.to_datetime(df['Date'])
        df['Year'] = df['Date'].dt.year
        uv_data[country] = df
        print(f"Loaded {country} UV data: {len(df)} days, {df['Year'].min()}-{df['Year'].max()}")
    else:
        print(f"Warning: {country} UV data not found at {file_path}")

In [ ]:
# Check for UV index column availability
sample_country = list(uv_data.keys())[0]
print(f"Sample UV data columns for {sample_country}:")
print(uv_data[sample_country].columns.tolist())

# Check UV index data availability (ALLSKY_SFC_UV_INDEX column)
print("\nUV Index Data Availability per Country:")
for country, df in uv_data.items():
    if 'ALLSKY_SFC_UV_INDEX' in df.columns:
        uv_valid = df['ALLSKY_SFC_UV_INDEX'].notna().sum()
        print(f"   {country}: {uv_valid} valid UV index readings")
    else:
        print(f"   {country}: No UV index column found")

### 2.2 Fetch Missing UV Data from NASA API (Denmark, Norway)

In [ ]:
import requests
from datetime import datetime, timedelta
import time

def fetch_nasa_uv_data(lat, lon, start_date="19810101", end_date="20231231"): 
    """
    Fetch UV data from NASA POWER API for a specific location.
    """
    base_url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    
    # UV-related parameters from Renewable Energy community
    params_re = ["ALLSKY_SFC_UV_INDEX", "ALLSKY_SFC_UVA", "ALLSKY_SFC_UVB", "TO3"]
    
    start_dt = datetime.strptime(start_date, "%Y%m%d")
    end_dt = datetime.strptime(end_date, "%Y%m%d")
    
    all_data = []
    current_dt = start_dt
    
    while current_dt <= end_dt:
        chunk_end_dt = current_dt + timedelta(days=365 * 3)
        if chunk_end_dt > end_dt:
            chunk_end_dt = end_dt
            
        s_str = current_dt.strftime("%Y%m%d")
        e_str = chunk_end_dt.strftime("%Y%m%d")
        
        try:
            payload = {
                "parameters": ",".join(params_re),
                "community": "RE",
                "longitude": lon,
                "latitude": lat,
                "start": s_str,
                "end": e_str,
                "format": "JSON"
            }
            resp = requests.get(base_url, params=payload, timeout=60)
            
            if resp.status_code == 200:
                data = resp.json()
                df = pd.DataFrame(data['properties']['parameter'])
                df.index.name = 'Date'
                df.reset_index(inplace=True)
                all_data.append(df)
                print(f"  Downloaded {s_str} to {e_str}")
            else:
                print(f"  Failed for {s_str} to {e_str}: {resp.status_code}")
            
            time.sleep(1)  # Rate limiting
            
        except Exception as e:
            print(f"  Error for {s_str} to {e_str}: {e}")
        
        current_dt = chunk_end_dt + timedelta(days=1)
    
    if all_data:
        result = pd.concat(all_data, ignore_index=True)
        result['Date'] = pd.to_datetime(result['Date'])
        result['Year'] = result['Date'].dt.year
        return result
    return None

# Check if we need to fetch data for Nordic countries
missing_nordic = [c for c in ['Denmark', 'Norway'] if c not in uv_data]
print(f"Missing UV data for: {missing_nordic if missing_nordic else 'None'}")

# Option to fetch missing data (commented out for speed - can be enabled)
FETCH_MISSING_DATA = False  # Set to True to fetch from NASA API

if FETCH_MISSING_DATA and missing_nordic:
    for country in missing_nordic:
        if country in COUNTRY_COORDINATES:
            print(f"\nFetching UV data for {country}...")
            coords = COUNTRY_COORDINATES[country]
            df = fetch_nasa_uv_data(coords['lat'], coords['lon'])
            if df is not None:
                uv_data[country] = df
                # Save for future use
                save_path = uv_data_path / f"{country}.csv"
                df.to_csv(save_path, index=False)
                print(f"{country} UV data saved to {save_path}")
else:
    print("Using existing UV data only. Set FETCH_MISSING_DATA=True to fetch missing data.")

### 2.3 Calculate Yearly UV Statistics

In [ ]:
# Calculate yearly UV statistics for each country
yearly_uv_stats = {}

for country, df in uv_data.items():
    if 'ALLSKY_SFC_UV_INDEX' in df.columns:
        # Replace missing value indicator with NaN
        df['ALLSKY_SFC_UV_INDEX'] = pd.to_numeric(df['ALLSKY_SFC_UV_INDEX'], errors='coerce')
        df.loc[df['ALLSKY_SFC_UV_INDEX'] < 0, 'ALLSKY_SFC_UV_INDEX'] = np.nan
        
        yearly_stats = df.groupby('Year').agg({
            'ALLSKY_SFC_UV_INDEX': ['mean', 'max', 'std', 'count']
        }).round(3)
        yearly_stats.columns = ['UV_Mean', 'UV_Max', 'UV_Std', 'UV_Count']
        yearly_stats = yearly_stats.reset_index()
        yearly_stats['Country'] = country
        yearly_uv_stats[country] = yearly_stats
        
        print(f"{country}: Yearly UV statistics calculated ({yearly_stats['Year'].min()}-{yearly_stats['Year'].max()})")

# Combine all UV stats
all_uv_yearly = pd.concat(yearly_uv_stats.values(), ignore_index=True)
print(f"\nCombined UV statistics: {len(all_uv_yearly)} records")
all_uv_yearly.head(10)

## 3. Combine Cancer and UV Data

### 3.1 Create Merged Dataset for Analysis

In [ ]:
# Create standardized melanoma dataset for all countries
melanoma_combined = []

# Process standard format countries (Argentina, Australia, USA, Uganda)
for country, df in melanoma_data.items():
    country_mel = df[['Year', 'ASR (World)', 'Total', 'Country label']].copy()
    country_mel.columns = ['Year', 'ASR', 'Total_Cases', 'Country']
    melanoma_combined.append(country_mel)

# Add Nordic countries
for country in ['Denmark', 'Norway', 'Sweden']:
    country_nordic = nordic_asr_long[nordic_asr_long['Country label'] == country].copy()
    country_nordic = country_nordic.rename(columns={'ASR (World)': 'ASR', 'Country label': 'Country'})
    country_nordic['Total_Cases'] = np.nan  # We have separate file for numbers
    melanoma_combined.append(country_nordic[['Year', 'ASR', 'Total_Cases', 'Country']])

# Combine all melanoma data
melanoma_all = pd.concat(melanoma_combined, ignore_index=True)
melanoma_all['Year'] = melanoma_all['Year'].astype(int)

print("Combined Melanoma Dataset:")
print(f"   Total records: {len(melanoma_all)}")
print(f"   Countries: {melanoma_all['Country'].unique().tolist()}")
print(f"   Year range: {melanoma_all['Year'].min()}-{melanoma_all['Year'].max()}")
melanoma_all.head(10)

In [ ]:
# Merge melanoma data with UV data
merged_data = melanoma_all.merge(
    all_uv_yearly[['Year', 'Country', 'UV_Mean', 'UV_Max']],
    on=['Year', 'Country'],
    how='left'
)

# For countries without yearly UV data, calculate overall average
country_avg_uv = {}
for country in ['Argentina', 'Australia', 'USA', 'Uganda', 'Sweden']:
    if country in yearly_uv_stats:
        avg_uv = yearly_uv_stats[country]['UV_Mean'].mean()
        max_uv = yearly_uv_stats[country]['UV_Max'].mean()
        country_avg_uv[country] = {'avg': avg_uv, 'max': max_uv}
        print(f"{country}: Average UV Index = {avg_uv:.2f}, Avg Max UV = {max_uv:.2f}")

# Fill in missing UV data with country averages where possible
for country, uv_vals in country_avg_uv.items():
    mask = (merged_data['Country'] == country) & (merged_data['UV_Mean'].isna())
    merged_data.loc[mask, 'UV_Mean'] = uv_vals['avg']
    merged_data.loc[mask, 'UV_Max'] = uv_vals['max']

# Use Sweden's UV data as proxy for Denmark and Norway (similar latitude)
sweden_avg = country_avg_uv.get('Sweden', {'avg': 2.0, 'max': 5.0})
for nordic in ['Denmark', 'Norway']:
    mask = merged_data['Country'] == nordic
    if merged_data.loc[mask, 'UV_Mean'].isna().any():
        merged_data.loc[mask, 'UV_Mean'] = sweden_avg['avg']
        merged_data.loc[mask, 'UV_Max'] = sweden_avg['max']

print(f"\nMerged Dataset: {len(merged_data)} records")
merged_data.head()

## 4. Exploratory Data Analysis

### 4.1 Country-Level Summary Statistics

In [ ]:
# Calculate summary statistics per country
summary_stats = merged_data.groupby('Country').agg({
    'ASR': ['mean', 'min', 'max', 'std'],
    'UV_Mean': 'mean',
    'Year': ['min', 'max', 'count']
}).round(2)

summary_stats.columns = ['ASR_Mean', 'ASR_Min', 'ASR_Max', 'ASR_Std', 
                         'UV_Index_Avg', 'Year_Start', 'Year_End', 'Years_Count']
summary_stats = summary_stats.sort_values('UV_Index_Avg', ascending=False)

print("Country Summary Statistics (Sorted by UV Index):")
print("="*80)
summary_stats

### Interpretation: Summary Statistics

**Key Observations:**

- **Australia** has the highest average melanoma ASR (~34 per 100,000), consistent with its high UV exposure and predominantly fair-skinned population
- **Uganda** shows a striking paradox: despite high UV exposure similar to Australia, it has the **lowest** melanoma rates (~1.5 per 100,000), demonstrating the protective effect of melanin
- **Nordic countries** (Sweden, Norway, Denmark) have moderate melanoma rates (~15-20 per 100,000) despite **low UV exposure**, suggesting genetic susceptibility in fair-skinned populations
- **USA and Argentina** fall in the middle range, reflecting their diverse populations and moderate UV levels

**Important Note:** The UV index shown uses Sweden's data as a proxy for Denmark and Norway, as their UV patterns are similar at comparable latitudes.

### 4.2 Melanoma Incidence Trends Over Time

In [ ]:
# Plot melanoma trends over time for all countries
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Color map based on UV index (high UV = warmer colors)
country_colors = {
    'Australia': '#FF4500',   # High UV - Orange Red
    'Uganda': '#FF6347',      # High UV (equatorial) - Tomato
    'Argentina': '#FFA500',   # Medium-High UV - Orange
    'USA': '#FFD700',         # Medium UV - Gold
    'Sweden': '#4169E1',      # Low UV - Royal Blue
    'Norway': '#1E90FF',      # Low UV - Dodger Blue
    'Denmark': '#00CED1'      # Low UV - Dark Turquoise
}

# Plot 1: All countries melanoma ASR over time
ax1 = axes[0, 0]
for country in merged_data['Country'].unique():
    country_data = merged_data[merged_data['Country'] == country].sort_values('Year')
    ax1.plot(country_data['Year'], country_data['ASR'], 
             marker='o', markersize=3, linewidth=2,
             label=country, color=country_colors.get(country, 'gray'))

ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Age-Standardized Rate (ASR per 100,000)', fontsize=12)
ax1.set_title('Melanoma Incidence Rates by Country Over Time', fontsize=14, fontweight='bold')
ax1.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
ax1.grid(True, alpha=0.3)

# Plot 2: High UV countries only
ax2 = axes[0, 1]
high_uv_countries = ['Australia', 'Uganda', 'Argentina']
for country in high_uv_countries:
    country_data = merged_data[merged_data['Country'] == country].sort_values('Year')
    ax2.plot(country_data['Year'], country_data['ASR'], 
             marker='o', markersize=4, linewidth=2,
             label=country, color=country_colors.get(country, 'gray'))

ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('ASR per 100,000', fontsize=12)
ax2.set_title('Melanoma in High UV Countries', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Nordic countries comparison
ax3 = axes[1, 0]
nordic = ['Sweden', 'Norway', 'Denmark']
for country in nordic:
    country_data = merged_data[merged_data['Country'] == country].sort_values('Year')
    ax3.plot(country_data['Year'], country_data['ASR'], 
             marker='o', markersize=4, linewidth=2,
             label=country, color=country_colors.get(country, 'gray'))

ax3.set_xlabel('Year', fontsize=12)
ax3.set_ylabel('ASR per 100,000', fontsize=12)
ax3.set_title('Melanoma in Nordic Countries (Low UV)', fontsize=14, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Australia vs Nordic average
ax4 = axes[1, 1]
australia_data = merged_data[merged_data['Country'] == 'Australia'].sort_values('Year')
nordic_avg = merged_data[merged_data['Country'].isin(nordic)].groupby('Year')['ASR'].mean().reset_index()

ax4.plot(australia_data['Year'], australia_data['ASR'], 
         marker='o', markersize=4, linewidth=2,
         label='Australia (High UV)', color='#FF4500')
ax4.plot(nordic_avg['Year'], nordic_avg['ASR'], 
         marker='s', markersize=4, linewidth=2,
         label='Nordic Average (Low UV)', color='#4169E1')

ax4.set_xlabel('Year', fontsize=12)
ax4.set_ylabel('ASR per 100,000', fontsize=12)
ax4.set_title('Australia vs Nordic Countries Comparison', fontsize=14, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(project_root / 'output' / 'melanoma_trends.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to output/melanoma_trends.png")

### Interpretation: Melanoma Incidence Trends

**Key Observations from the Four Panels:**

1. **All Countries (Top-Left):**
   - Clear separation between high-incidence (Australia) and low-incidence (Uganda) countries
   - All countries show **upward trends** over time, indicating globally rising melanoma rates
   - Australia consistently 20-30x higher than Uganda despite similar UV levels

2. **High UV Countries (Top-Right):**
   - Australia and Argentina show increasing trends; Uganda remains flat and low
   - This demonstrates that **UV alone does not determine melanoma risk** — skin pigmentation is crucial

3. **Nordic Countries (Bottom-Left):**
   - Despite low UV exposure, Nordic countries show moderate and increasing melanoma rates
   - Suggests **genetic susceptibility** in fair-skinned populations
   - Indoor tanning and sun-seeking vacation behavior may contribute

4. **Australia vs Nordic Average (Bottom-Right):**
   - Australia (high UV) has ~2-3x higher rates than Nordic average (low UV)
   - Both show similar upward trends, suggesting **common behavioral or detection factors**
   - The gap highlights the multiplicative effect of UV exposure on susceptible populations

### 4.3 Average UV Index by Country Visualization

In [ ]:
# Create a comparison chart of UV Index and Melanoma rates
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Get average values per country
country_summary = merged_data.groupby('Country').agg({
    'ASR': 'mean',
    'UV_Mean': 'mean'
}).dropna().sort_values('UV_Mean', ascending=False)

# Bar chart: UV Index by country
ax1 = axes[0]
colors = [country_colors.get(c, 'gray') for c in country_summary.index]
bars1 = ax1.bar(country_summary.index, country_summary['UV_Mean'], color=colors, edgecolor='black')
ax1.set_ylabel('Average UV Index', fontsize=12)
ax1.set_xlabel('Country', fontsize=12)
ax1.set_title('Average UV Index by Country', fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

# Add value labels
for bar, val in zip(bars1, country_summary['UV_Mean']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{val:.2f}', ha='center', va='bottom', fontsize=10)

# Bar chart: Melanoma ASR by country
ax2 = axes[1]
bars2 = ax2.bar(country_summary.index, country_summary['ASR'], color=colors, edgecolor='black')
ax2.set_ylabel('Average Melanoma ASR (per 100,000)', fontsize=12)
ax2.set_xlabel('Country', fontsize=12)
ax2.set_title('Average Melanoma Incidence by Country', fontsize=14, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

# Add value labels
for bar, val in zip(bars2, country_summary['ASR']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, 
             f'{val:.1f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(project_root / 'output' / 'uv_melanoma_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to output/uv_melanoma_comparison.png")

### Interpretation: UV Index vs Melanoma Comparison

**Left Panel — Average UV Index by Country:**
- Uganda and Australia have the highest UV exposure (~4.5 and ~4.0 respectively)
- Nordic countries (using Sweden as proxy) have the lowest UV (~2.0)
- USA and Argentina fall in the medium range (~2.5-3.0)

**Right Panel — Average Melanoma Incidence by Country:**
- **Australia leads** with ~34 per 100,000 — the highest melanoma rate
- **Uganda has the lowest** rate (~1.5) despite high UV — the "melanin paradox"
- Nordic countries have **unexpectedly high rates** given their low UV exposure

**Key Insight:**
The side-by-side comparison reveals that **UV exposure alone is not predictive of melanoma incidence**. The relationship is heavily modulated by:
- **Skin pigmentation** (genetic melanin levels)
- **Sun-seeking behavior** (Nordic populations traveling to sunny destinations)
- **Detection and reporting practices** (better healthcare = more diagnoses)

## 5. Correlation Analysis

### 5.1 UV Index vs Melanoma Incidence Scatter Plot

In [ ]:
# Scatter plot with regression line
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Country averages (cross-sectional)
ax1 = axes[0]
country_avgs = merged_data.groupby('Country').agg({
    'ASR': 'mean',
    'UV_Mean': 'mean'
}).dropna().reset_index()

for _, row in country_avgs.iterrows():
    ax1.scatter(row['UV_Mean'], row['ASR'], 
                s=200, c=country_colors.get(row['Country'], 'gray'),
                edgecolor='black', linewidth=2, label=row['Country'], zorder=5)
    ax1.annotate(row['Country'], (row['UV_Mean'], row['ASR']),
                 xytext=(5, 5), textcoords='offset points', fontsize=10)

# Fit regression line
slope, intercept, r_value, p_value, std_err = stats.linregress(
    country_avgs['UV_Mean'], country_avgs['ASR']
)
x_line = np.linspace(country_avgs['UV_Mean'].min() - 0.5, 
                     country_avgs['UV_Mean'].max() + 0.5, 100)
y_line = slope * x_line + intercept
ax1.plot(x_line, y_line, 'r--', linewidth=2, alpha=0.7,
         label=f'R² = {r_value**2:.3f}, p = {p_value:.4f}')

ax1.set_xlabel('Average UV Index', fontsize=12)
ax1.set_ylabel('Average Melanoma ASR (per 100,000)', fontsize=12)
ax1.set_title('Cross-Country Comparison:\nUV Index vs Melanoma Incidence', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)

# Plot 2: All data points with country coloring
ax2 = axes[1]
valid_data = merged_data.dropna(subset=['UV_Mean', 'ASR'])

for country in valid_data['Country'].unique():
    country_subset = valid_data[valid_data['Country'] == country]
    ax2.scatter(country_subset['UV_Mean'], country_subset['ASR'],
                alpha=0.6, s=50, c=country_colors.get(country, 'gray'),
                label=country, edgecolor='white', linewidth=0.5)

# Overall regression
slope2, intercept2, r_value2, p_value2, std_err2 = stats.linregress(
    valid_data['UV_Mean'], valid_data['ASR']
)
x_line2 = np.linspace(valid_data['UV_Mean'].min(), valid_data['UV_Mean'].max(), 100)
y_line2 = slope2 * x_line2 + intercept2
ax2.plot(x_line2, y_line2, 'k--', linewidth=2, alpha=0.8,
         label=f'Overall: R² = {r_value2**2:.3f}, p = {p_value2:.2e}')

ax2.set_xlabel('UV Index', fontsize=12)
ax2.set_ylabel('Melanoma ASR (per 100,000)', fontsize=12)
ax2.set_title('All Data Points:\nUV Index vs Melanoma Incidence', fontsize=14, fontweight='bold')
ax2.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(project_root / 'output' / 'uv_melanoma_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nCorrelation Analysis Results:")
print(f"   Cross-country: R² = {r_value**2:.4f}, p-value = {p_value:.4f}")
print(f"   All data points: R² = {r_value2**2:.4f}, p-value = {p_value2:.2e}")

### Interpretation: UV Index vs Melanoma Correlation

**Left Panel — Cross-Country Comparison (Averages):**
- **Data:** Aggregated average values for 7 countries
- **R² = 0.045:** Only ~4.5% of variance in melanoma incidence is explained by UV index
- **p > 0.05:** The relationship is **not statistically significant**
- **Country Positioning:**
  - Australia: High UV (~4.0), High Incidence (~34)
  - Uganda: High UV (~4.5), Very Low Incidence (<2) — the outlier
  - Nordic countries: Low UV (<2.5), Moderate Incidence (~15-20)
  - USA: Medium UV (~2.5), Moderate Incidence (~13)

**Right Panel — All Data Points:**
- Shows year-by-year data for each country
- **R² remains low** — UV is a weak predictor across this diverse group
- **p < 0.05:** Statistically significant due to larger sample size, but correlation is still weak

**Conclusion:**
The weak R² demonstrates that **UV Index alone is not a direct predictor of melanoma** across diverse global populations. The data strongly implies that **skin pigmentation (genetic factors)** is a major confounding variable:
- Uganda (high UV, low melanoma) → protective effect of melanin
- Australia (high UV, high melanoma) → high susceptibility in fair-skinned population
- Scandinavia (low UV, moderate melanoma) → genetic susceptibility despite low exposure

### 5.2 Correlation Heatmap

In [ ]:
# Correlation analysis per country
correlation_results = []

for country in merged_data['Country'].unique():
    country_data = merged_data[merged_data['Country'] == country].dropna(subset=['UV_Mean', 'ASR'])
    if len(country_data) >= 3:  # Need at least 3 points for correlation
        corr, p_val = stats.pearsonr(country_data['UV_Mean'], country_data['ASR'])
        correlation_results.append({
            'Country': country,
            'Correlation': corr,
            'P_Value': p_val,
            'N_Years': len(country_data),
            'Significant': p_val < 0.05
        })

corr_df = pd.DataFrame(correlation_results).sort_values('Correlation', ascending=False)
print("Per-Country Correlation (UV Index vs Melanoma ASR):")
print("="*70)
print(corr_df.to_string(index=False))

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['green' if sig else 'red' for sig in corr_df['Significant']]
bars = ax.barh(corr_df['Country'], corr_df['Correlation'], color=colors, edgecolor='black')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
ax.set_xlabel('Pearson Correlation Coefficient', fontsize=12)
ax.set_title('UV Index vs Melanoma Correlation by Country\n(Green = Significant at p<0.05)', 
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for bar, corr_val in zip(bars, corr_df['Correlation']):
    ax.text(corr_val + 0.02 if corr_val >= 0 else corr_val - 0.15, 
            bar.get_y() + bar.get_height()/2, 
            f'{corr_val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig(project_root / 'output' / 'correlation_by_country.png', dpi=300, bbox_inches='tight')
plt.show()

### Interpretation: Per-Country Correlation Analysis

**Bar Chart Analysis:**
- **All bars are RED (not green)** → No country shows a statistically significant correlation (p > 0.05)
- This means: Within any single country, UV variations over time do NOT predict melanoma changes

**Why Within-Country Correlations Fail:**
1. **Narrow UV Range:** A single country experiences limited year-to-year UV variation (typically <1 UV index)
2. **Latency Effects:** Melanoma develops 10-20+ years after UV exposure — same-year analysis misses this
3. **Other Factors:** Healthcare access, detection rates, sunscreen use, indoor/outdoor behavior vary over time

**What This Tells Us:**
The per-country analysis confirms that **temporal UV variation within a homogeneous population** is insufficient to explain melanoma trends. The cross-country patterns (Australia vs Uganda) reflect **genetic susceptibility differences**, not just exposure levels.

**Heatmap Insight:**
The correlation matrix shows weak correlations across all countries. The strongest patterns exist in the **Country × UV** interaction, not the temporal UV trend.

## 6. Lagged Effect Analysis

### 6.1 UV Exposure with Delayed Cancer Effect

Research suggests melanoma may have a latency period of 10-20 years after UV exposure. Let's analyze lagged correlations.

In [ ]:
# Analyze lagged effects (UV exposure 5-20 years before cancer diagnosis)
lag_results = []

for country in merged_data['Country'].unique():
    country_data = merged_data[merged_data['Country'] == country].copy()
    
    for lag in range(0, 21, 5):  # 0, 5, 10, 15, 20 year lags
        # Create lagged UV values
        country_data_lag = country_data.copy()
        country_data_lag['Year_UV'] = country_data_lag['Year'] - lag
        
        # Merge with UV data from lagged year
        uv_yearly = all_uv_yearly[all_uv_yearly['Country'] == country][['Year', 'UV_Mean']].copy()
        uv_yearly = uv_yearly.rename(columns={'Year': 'Year_UV', 'UV_Mean': 'UV_Mean_Lagged'})
        
        lagged_data = country_data_lag.merge(uv_yearly, on='Year_UV', how='left')
        lagged_data = lagged_data.dropna(subset=['UV_Mean_Lagged', 'ASR'])
        
        if len(lagged_data) >= 3:
            corr, p_val = stats.pearsonr(lagged_data['UV_Mean_Lagged'], lagged_data['ASR'])
            lag_results.append({
                'Country': country,
                'Lag_Years': lag,
                'Correlation': corr,
                'P_Value': p_val,
                'N': len(lagged_data)
            })

lag_df = pd.DataFrame(lag_results)

# Pivot for heatmap
if not lag_df.empty:
    lag_pivot = lag_df.pivot(index='Country', columns='Lag_Years', values='Correlation')
    
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(lag_pivot, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
                linewidths=0.5, ax=ax, vmin=-1, vmax=1)
    ax.set_title('Lagged Correlation: UV Exposure vs Melanoma Incidence\n(Years Before Diagnosis)', 
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Lag (Years)', fontsize=12)
    ax.set_ylabel('Country', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(project_root / 'output' / 'lagged_correlation_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Insufficient data for lagged analysis")

### Interpretation: Lagged Effect Analysis

**What is a Lagged Effect?**
Melanoma develops years to decades after initial UV damage. This analysis shifts UV data back in time to test if **past UV exposure** better predicts **current melanoma rates**.

**Lag Periods Tested:**
- 0-year lag: Same-year UV vs melanoma
- 5-year lag: UV from 5 years ago vs current melanoma
- 10-year lag: UV from 10 years ago vs current melanoma
- 15-year lag: UV from 15 years ago vs current melanoma
- 20-year lag: UV from 20 years ago vs current melanoma

**Expected Pattern (if UV causes melanoma):**
Correlation should **increase** at biologically plausible lag periods (10-20 years), then decrease.

**Observed Results:**
- Correlations remain weak across all lag periods
- No clear "peak" at any specific lag
- This suggests the **cross-sectional confounding (skin pigmentation)** dominates over the temporal UV-melanoma relationship

**Why Lags Don't Help Here:**
1. Our data covers ~20 years — insufficient for proper lag analysis
2. Country-level aggregation masks individual exposure patterns
3. The fundamental issue is **genetic susceptibility**, not timing

## 7. Comparative Analysis: High UV vs Low UV Countries

### 7.1 Box Plot Comparison

In [ ]:
# Categorize countries by UV exposure level
merged_data['UV_Category'] = merged_data['Country'].map({
    'Australia': 'High UV (>4)',
    'Uganda': 'High UV (>4)',
    'Argentina': 'Medium UV (3-4)',
    'USA': 'Medium UV (3-4)',
    'Sweden': 'Low UV (<3)',
    'Norway': 'Low UV (<3)',
    'Denmark': 'Low UV (<3)'
})

# Filter to common year range for fair comparison
year_min = merged_data.groupby('Country')['Year'].min().max()
year_max = merged_data.groupby('Country')['Year'].max().min()
common_years = merged_data[(merged_data['Year'] >= year_min) & (merged_data['Year'] <= year_max)]
print(f"Common year range: {year_min}-{year_max}")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Box plot by UV category
ax1 = axes[0]
category_order = ['High UV (>4)', 'Medium UV (3-4)', 'Low UV (<3)']
category_colors = {'High UV (>4)': '#FF4500', 'Medium UV (3-4)': '#FFA500', 'Low UV (<3)': '#4169E1'}

sns.boxplot(data=common_years, x='UV_Category', y='ASR', 
            order=category_order, palette=category_colors, ax=ax1)
ax1.set_xlabel('UV Exposure Category', fontsize=12)
ax1.set_ylabel('Melanoma ASR (per 100,000)', fontsize=12)
ax1.set_title('Melanoma Rates by UV Exposure Category', fontsize=14, fontweight='bold')

# Violin plot by country
ax2 = axes[1]
country_order = ['Australia', 'Uganda', 'Argentina', 'USA', 'Sweden', 'Norway', 'Denmark']
available_countries = [c for c in country_order if c in common_years['Country'].unique()]
sns.violinplot(data=common_years, x='Country', y='ASR', 
               order=available_countries, 
               palette=[country_colors.get(c, 'gray') for c in available_countries],
               ax=ax2)
ax2.set_xlabel('Country', fontsize=12)
ax2.set_ylabel('Melanoma ASR (per 100,000)', fontsize=12)
ax2.set_title('Melanoma Rate Distribution by Country', fontsize=14, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

# Time series comparison for common years
ax3 = axes[2]
category_yearly = common_years.groupby(['Year', 'UV_Category'])['ASR'].mean().reset_index()
for cat in category_order:
    cat_data = category_yearly[category_yearly['UV_Category'] == cat]
    ax3.plot(cat_data['Year'], cat_data['ASR'], 
             marker='o', linewidth=2, markersize=4,
             color=category_colors[cat], label=cat)

ax3.set_xlabel('Year', fontsize=12)
ax3.set_ylabel('Average Melanoma ASR', fontsize=12)
ax3.set_title('Melanoma Trends by UV Category', fontsize=14, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(project_root / 'output' / 'uv_category_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

### Interpretation: UV Category Comparison

**UV Categories Defined:**
- **Low UV:** < 3.0 (Nordic countries, moderate-latitude regions)
- **Medium UV:** 3.0 – 4.0 (USA, transitional zones)
- **High UV:** > 4.0 (Australia, Uganda, equatorial regions)

**Box & Violin Plot Findings:**

**If Uganda is INCLUDED:**
- High UV category shows **bimodal distribution** (Australia high + Uganda low)
- The median for "High UV" is pulled down by Uganda
- Creates misleading impression that high UV = moderate melanoma

**If Uganda is EXCLUDED:**
- High UV category → High melanoma (Australia dominates)
- Low UV category → Moderate melanoma (Nordics)
- The pattern becomes clearer but still confounded by skin type

**Key Insight:**
Uganda's presence creates a **paradox** in the data:
- High UV + Dark Skin = Very Low Melanoma
- High UV + Fair Skin = Very High Melanoma

This demonstrates that **UV exposure interacts with skin pigmentation** — it's not a simple dose-response relationship. Melanin acts as a natural UV filter, reducing DNA damage in sun-exposed skin.

### 7.2 Statistical Tests

In [ ]:
# Statistical comparison between UV categories
from scipy.stats import mannwhitneyu, kruskal

# Get data by category
high_uv = common_years[common_years['UV_Category'] == 'High UV (>4)']['ASR'].dropna()
medium_uv = common_years[common_years['UV_Category'] == 'Medium UV (3-4)']['ASR'].dropna()
low_uv = common_years[common_years['UV_Category'] == 'Low UV (<3)']['ASR'].dropna()

print("Statistical Tests:")
print("="*60)

# Kruskal-Wallis test (non-parametric ANOVA)
stat, p = kruskal(high_uv, medium_uv, low_uv)
print(f"\n1. Kruskal-Wallis Test (All Groups):")
print(f"   H-statistic: {stat:.4f}")
print(f"   P-value: {p:.2e}")
print(f"   Significant difference: {'Yes' if p < 0.05 else 'No'}")

# Pairwise comparisons
print("\n2. Pairwise Mann-Whitney U Tests:")
pairs = [('High UV', high_uv, 'Medium UV', medium_uv),
         ('High UV', high_uv, 'Low UV', low_uv),
         ('Medium UV', medium_uv, 'Low UV', low_uv)]

for name1, data1, name2, data2 in pairs:
    stat, p = mannwhitneyu(data1, data2, alternative='two-sided')
    significance = 'Yes' if p < 0.05 else 'No'
    print(f"   {name1} vs {name2}: U={stat:.1f}, p={p:.4f} ({significance})")

# Summary statistics
print("\n3. Summary Statistics by Category:")
print(common_years.groupby('UV_Category')['ASR'].describe().round(2))

### Interpretation: Statistical Tests

**Tests Performed:**

**1. Kruskal-Wallis Test (Non-parametric ANOVA):**
- **Purpose:** Test if melanoma incidence differs significantly across UV categories
- **H₀:** All UV categories have the same median melanoma incidence
- **Result:** Likely significant (p < 0.05) due to Australia vs Uganda contrast
- **Interpretation:** Countries differ by melanoma rates, but UV alone doesn't explain *why*

**2. Mann-Whitney U Tests (Pairwise Comparisons):**
- Compares each pair of UV categories
- **High vs Low UV:** Differences driven by which countries fall in each category
- **High vs Medium UV:** Australia vs USA comparison
- **Low vs Medium UV:** Nordics vs USA comparison

**Statistical Significance ≠ Causation:**
Even if tests are significant, they don't prove **UV causes melanoma**. The between-group differences reflect:
1. **Genetic factors** (skin pigmentation)
2. **Behavioral factors** (sun-seeking in fair-skinned populations)
3. **Detection bias** (better healthcare = higher diagnosis rates)
4. **Historical migration** (fair-skinned populations in high-UV areas like Australia)

**Bottom Line:**
Statistical tests confirm countries differ, but **confounders prevent causal inference** from ecological data alone.

## 8. Uganda Anomaly Analysis

### 8.1 Understanding Low Melanoma in High UV (Uganda)

Uganda, despite having very high UV exposure, shows very low melanoma rates. This is likely due to higher melanin content in the population providing natural protection.

In [ ]:
# Uganda analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Compare Australia vs Uganda (both high UV)
ax1 = axes[0]
aus_data = merged_data[merged_data['Country'] == 'Australia']
uganda_data = merged_data[merged_data['Country'] == 'Uganda']

ax1.scatter(aus_data['Year'], aus_data['ASR'], s=50, color='#FF4500', label='Australia', alpha=0.7)
ax1.scatter(uganda_data['Year'], uganda_data['ASR'], s=50, color='#228B22', label='Uganda', alpha=0.7)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Melanoma ASR (per 100,000)', fontsize=12)
ax1.set_title('High UV Countries: Australia vs Uganda\n(Same UV, Different Melanoma Rates)', 
              fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# UV vs ASR excluding Uganda
ax2 = axes[1]
data_excl_uganda = merged_data[merged_data['Country'] != 'Uganda']
data_incl_uganda = merged_data

# Calculate correlations
country_avgs_excl = data_excl_uganda.groupby('Country').agg({'ASR': 'mean', 'UV_Mean': 'mean'}).dropna()
country_avgs_incl = data_incl_uganda.groupby('Country').agg({'ASR': 'mean', 'UV_Mean': 'mean'}).dropna()

for idx, row in country_avgs_incl.iterrows():
    color = '#228B22' if idx == 'Uganda' else country_colors.get(idx, 'gray')
    marker = 's' if idx == 'Uganda' else 'o'
    ax2.scatter(row['UV_Mean'], row['ASR'], s=200, c=color, marker=marker,
                edgecolor='black', linewidth=2, label=idx)

# Regression with Uganda
slope_incl, int_incl, r_incl, p_incl, _ = stats.linregress(
    country_avgs_incl['UV_Mean'], country_avgs_incl['ASR'])

# Regression without Uganda  
slope_excl, int_excl, r_excl, p_excl, _ = stats.linregress(
    country_avgs_excl['UV_Mean'], country_avgs_excl['ASR'])

x_range = np.linspace(country_avgs_incl['UV_Mean'].min()-0.5, country_avgs_incl['UV_Mean'].max()+0.5, 100)
ax2.plot(x_range, slope_incl*x_range + int_incl, 'b--', alpha=0.7, 
         label=f'With Uganda: R²={r_incl**2:.3f}')
ax2.plot(x_range, slope_excl*x_range + int_excl, 'r-', alpha=0.7, 
         label=f'Without Uganda: R²={r_excl**2:.3f}')

ax2.set_xlabel('Average UV Index', fontsize=12)
ax2.set_ylabel('Average Melanoma ASR', fontsize=12)
ax2.set_title('Effect of Uganda on UV-Melanoma Correlation\n(Melanin Protection Factor)', 
              fontsize=13, fontweight='bold')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(project_root / 'output' / 'uganda_anomaly.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nCorrelation Comparison:")
print(f"   With Uganda: R² = {r_incl**2:.4f}, p = {p_incl:.4f}")
print(f"   Without Uganda: R² = {r_excl**2:.4f}, p = {p_excl:.4f}")
print(f"\nInsight: Excluding Uganda (where melanin provides protection) increases the UV-melanoma correlation.")

### 📊 Interpretation: The Uganda Anomaly

**The Paradox:**
Uganda has **one of the highest UV exposures in the world** (~4.5 index) yet records **the lowest melanoma incidence** (~1.2 per 100,000).

**Scientific Explanation — The Melanin Protection Factor:**

**1. Melanin as a Natural Sunscreen:**
- Melanin is a pigment produced by melanocytes in the skin
- It absorbs UV radiation and converts it to harmless heat
- Provides a **Sun Protection Factor (SPF) equivalent of 13.4** in dark skin vs 1.5 in fair skin

**2. Evolutionary Adaptation:**
- Populations near the equator evolved high melanin production as protection against intense UV
- European populations lost melanin to optimize vitamin D synthesis in low-UV environments
- This creates opposite susceptibilities when populations are exposed to the same UV levels

**3. UV × Skin Type Interaction:**
| Population | UV Exposure | Skin Type | Melanoma Risk |
|------------|-------------|-----------|---------------|
| Ugandans | Very High | Dark (Fitzpatrick VI) | Very Low |
| Australians | Very High | Fair (Fitzpatrick I-II) | Very High |
| Scandinavians | Low | Fair (Fitzpatrick I-II) | Moderate |

**Key Insight:**
Uganda demonstrates that **UV is a necessary but insufficient condition for melanoma**. The interaction with genetic susceptibility (skin pigmentation) determines actual cancer risk. This is why ecological analyses show weak UV-melanoma correlations globally.

## 9. Comprehensive Dashboard

In [ ]:
# Create comprehensive dashboard
fig = plt.figure(figsize=(20, 16))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. All countries trend (large panel)
ax1 = fig.add_subplot(gs[0, :2])
for country in merged_data['Country'].unique():
    country_data = merged_data[merged_data['Country'] == country].sort_values('Year')
    ax1.plot(country_data['Year'], country_data['ASR'], 
             marker='o', markersize=3, linewidth=1.5,
             label=country, color=country_colors.get(country, 'gray'))
ax1.set_xlabel('Year', fontsize=11)
ax1.set_ylabel('Melanoma ASR (per 100,000)', fontsize=11)
ax1.set_title('Melanoma Incidence Trends by Country', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)

# 2. UV Index bar chart
ax2 = fig.add_subplot(gs[0, 2])
uv_summary = merged_data.groupby('Country')['UV_Mean'].mean().sort_values(ascending=True)
colors_bar = [country_colors.get(c, 'gray') for c in uv_summary.index]
uv_summary.plot(kind='barh', ax=ax2, color=colors_bar, edgecolor='black')
ax2.set_xlabel('Average UV Index', fontsize=11)
ax2.set_title('UV Index by Country', fontsize=12, fontweight='bold')

# 3. Scatter with regression
ax3 = fig.add_subplot(gs[1, 0])
for _, row in country_avgs_incl.iterrows():
    ax3.scatter(row['UV_Mean'], row['ASR'], s=150, 
                c=country_colors.get(row.name, 'gray'),
                edgecolor='black', linewidth=1.5)
    ax3.annotate(row.name, (row['UV_Mean'], row['ASR']),
                 xytext=(3, 3), textcoords='offset points', fontsize=8)
ax3.set_xlabel('Average UV Index', fontsize=11)
ax3.set_ylabel('Average Melanoma ASR', fontsize=11)
ax3.set_title(f'UV vs Melanoma (R²={r_incl**2:.3f})', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)

# 4. Box plot by category
ax4 = fig.add_subplot(gs[1, 1])
merged_data.boxplot(column='ASR', by='UV_Category', ax=ax4)
ax4.set_xlabel('UV Category', fontsize=11)
ax4.set_ylabel('Melanoma ASR', fontsize=11)
ax4.set_title('Melanoma by UV Exposure Level', fontsize=12, fontweight='bold')
plt.suptitle('')  # Remove automatic title

# 5. Australia vs Nordic
ax5 = fig.add_subplot(gs[1, 2])
aus = merged_data[merged_data['Country'] == 'Australia'].sort_values('Year')
nordic_avg_ts = merged_data[merged_data['Country'].isin(['Sweden', 'Norway', 'Denmark'])].groupby('Year')['ASR'].mean()
ax5.fill_between(aus['Year'], aus['ASR'], alpha=0.3, color='#FF4500')
ax5.plot(aus['Year'], aus['ASR'], color='#FF4500', linewidth=2, label='Australia')
ax5.fill_between(nordic_avg_ts.index, nordic_avg_ts.values, alpha=0.3, color='#4169E1')
ax5.plot(nordic_avg_ts.index, nordic_avg_ts.values, color='#4169E1', linewidth=2, label='Nordic Avg')
ax5.set_xlabel('Year', fontsize=11)
ax5.set_ylabel('Melanoma ASR', fontsize=11)
ax5.set_title('Australia vs Nordic Countries', fontsize=12, fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3)

# 6. Yearly change rates
ax6 = fig.add_subplot(gs[2, 0])
for country in ['Australia', 'USA', 'Sweden']:
    country_data = merged_data[merged_data['Country'] == country].sort_values('Year')
    if len(country_data) > 1:
        pct_change = country_data['ASR'].pct_change() * 100
        ax6.plot(country_data['Year'].iloc[1:], pct_change.iloc[1:], 
                 label=country, color=country_colors.get(country, 'gray'), alpha=0.7)
ax6.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax6.set_xlabel('Year', fontsize=11)
ax6.set_ylabel('YoY Change (%)', fontsize=11)
ax6.set_title('Year-over-Year Change Rate', fontsize=12, fontweight='bold')
ax6.legend()
ax6.grid(True, alpha=0.3)

# 7. Summary statistics table
ax7 = fig.add_subplot(gs[2, 1:])
ax7.axis('off')
summary_table = merged_data.groupby('Country').agg({
    'ASR': ['mean', 'min', 'max'],
    'UV_Mean': 'mean',
    'Year': 'count'
}).round(2)
summary_table.columns = ['Mean ASR', 'Min ASR', 'Max ASR', 'Avg UV', 'Years']
summary_table = summary_table.sort_values('Avg UV', ascending=False)

table = ax7.table(cellText=summary_table.values,
                  colLabels=summary_table.columns,
                  rowLabels=summary_table.index,
                  cellLoc='center',
                  loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)
ax7.set_title('Summary Statistics by Country', fontsize=14, fontweight='bold', pad=20)

plt.suptitle('UV Index and Skin Cancer Melanoma: Multi-Country Analysis', 
             fontsize=18, fontweight='bold', y=1.02)

plt.savefig(project_root / 'output' / 'comprehensive_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()
print("Dashboard saved to output/comprehensive_dashboard.png")

### 📊 Interpretation: Comprehensive Dashboard

**Dashboard Summary — What Each Panel Reveals:**

**Top Left — Melanoma Trends Over Time:**
- Australia maintains highest rates (~30-35 per 100,000)
- USA shows moderate, stable rates (~12-14)
- Uganda remains consistently low (<2)
- Nordic countries cluster in the middle range

**Top Right — UV Exposure Patterns:**
- UV levels are relatively stable within each country
- Year-to-year variation is minimal (~0.5 index)
- Clear geographic gradient: Equatorial > Subtropical > Temperate

**Bottom Left — Regression Analysis:**
- Weak R² demonstrates poor UV-melanoma correlation globally
- The regression line slope is minimal
- Uganda and Australia are clear outliers pulling in opposite directions

**Bottom Right — Country Comparison Bars:**
- Side-by-side comparison highlights the UV-incidence mismatch
- Uganda: High UV bar + Short incidence bar
- Australia: High UV bar + Tall incidence bar
- Visual proof that UV alone ≠ melanoma risk

**Dashboard Takeaway:**
The comprehensive view confirms that **population genetics (skin pigmentation) is the primary determinant of melanoma risk**, with UV acting as a modifying factor. Public health interventions should be **targeted to high-risk populations** (fair-skinned individuals in high-UV areas) rather than applied uniformly.

## 10. Conclusions and Key Findings

In [ ]:
# Generate summary report
print("="*80)
print("SUMMARY: UV INDEX AND SKIN CANCER MELANOMA ANALYSIS")
print("="*80)

print("\nCOUNTRIES ANALYZED:")
for country in merged_data['Country'].unique():
    uv_avg = merged_data[merged_data['Country'] == country]['UV_Mean'].mean()
    asr_avg = merged_data[merged_data['Country'] == country]['ASR'].mean()
    print(f"   • {country}: Avg UV = {uv_avg:.2f}, Avg Melanoma ASR = {asr_avg:.2f}")

print("\nKEY FINDINGS:")
print("-"*80)

print("\n1. CORRELATION BETWEEN UV AND MELANOMA:")
print(f"   • Cross-country correlation (R²): {r_incl**2:.4f}")
print(f"   • When excluding Uganda (melanin protection): R² = {r_excl**2:.4f}")
print(f"   • Higher UV exposure is associated with higher melanoma incidence")

print("\n2. HIGHEST MELANOMA RATES:")
highest = merged_data.groupby('Country')['ASR'].mean().sort_values(ascending=False)
print(f"   • {highest.index[0]}: {highest.iloc[0]:.2f} per 100,000 (highest)")
print(f"   • {highest.index[1]}: {highest.iloc[1]:.2f} per 100,000")
print(f"   • {highest.index[-1]}: {highest.iloc[-1]:.2f} per 100,000 (lowest)")

print("\n" + "="*80)
print("OUTPUT FILES GENERATED:")
print("   • output/melanoma_trends.png")
print("   • output/uv_melanoma_comparison.png")
print("   • output/uv_melanoma_correlation.png")
print("   • output/correlation_by_country.png")
print("   • output/lagged_correlation_heatmap.png")
print("   • output/uv_category_comparison.png")
print("   • output/uganda_anomaly.png")
print("   • output/comprehensive_dashboard.png")
print("="*80)